# 03 — Score & baselines (Colab)

Rescore existing plan-executor predictions or run **programmatic baselines** without reloading the LLM.

Restores dataset and results from Drive (notebook **02**). **CPU is enough.**

Reports both **`overall`** (item micro) and **`composition_macro`** (equal weight per composition).

In [ ]:
# @title Configuration
MUSICINSTRUCT_REPO = "https://github.com/juliagsy/musicinstruct.git"
BRANCH = "main"  # @param {type:"string"}

WORK = "/content"
MUSICINSTRUCT = f"{WORK}/musicinstruct"

DATASET_MODE = "real_v0.2"  # @param ["pilot", "real_v0.2"]
RUN_ID = "real_v0.2_v1"  # @param {type:"string"}  use pilot_v1 for pilot (already scored — do not rerun)
SPLIT = "test"  # @param ["test", "validation"]

LOCAL_DATA = f"{MUSICINSTRUCT}/data/pilot" if DATASET_MODE == "pilot" else f"{MUSICINSTRUCT}/data/v0.2"
MANIFEST_NAME = "pilot.jsonl" if DATASET_MODE == "pilot" else "manifest.jsonl"
MANIFEST = f"{LOCAL_DATA}/{MANIFEST_NAME}"

USE_DRIVE = True  # @param {type:"boolean"}
DRIVE_ROOT = "/content/drive/MyDrive/musicinstruct"
DRIVE_DATA = f"{DRIVE_ROOT}/data/pilot" if DATASET_MODE == "pilot" else f"{DRIVE_ROOT}/data/v0.2"
DRIVE_RESULTS = f"{DRIVE_ROOT}/runs/{RUN_ID}/plan_executor_llama"
RESULTS_SUBDIR = "plan_executor_llama"  # @param {type:"string"}
LOCAL_RESULTS = f"{MUSICINSTRUCT}/results/{RESULTS_SUBDIR}"
DRIVE_BASELINES = f"{DRIVE_ROOT}/runs/{RUN_ID}/baselines"

RUN_BASELINES = False  # @param {type:"boolean"}  set True only to regenerate baselines
RESTORE_BASELINES = True  # @param {type:"boolean"}  copy existing baselines from Drive
RESCORE_PLAN_EXECUTOR = True  # @param {type:"boolean"}

In [ ]:
# @title Bootstrap & restore from Drive
import os
import shutil
import subprocess
from pathlib import Path

if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")

if Path(MUSICINSTRUCT).is_dir():
    shutil.rmtree(MUSICINSTRUCT)
subprocess.run(
    f"git clone --branch {BRANCH} --depth 1 {MUSICINSTRUCT_REPO} {MUSICINSTRUCT}",
    shell=True,
    check=True,
)
os.chdir(MUSICINSTRUCT)
subprocess.run("pip install -q -e .", shell=True, check=True)

if USE_DRIVE:
    drive_manifest = Path(DRIVE_DATA) / MANIFEST_NAME
    if not drive_manifest.is_file():
        raise FileNotFoundError(
            f"No {MANIFEST_NAME} on Drive at {DRIVE_DATA}. "
            f"Run notebook 01 with DATASET_MODE={DATASET_MODE!r} first."
        )
    Path(LOCAL_DATA).mkdir(parents=True, exist_ok=True)
    shutil.copytree(DRIVE_DATA, LOCAL_DATA, dirs_exist_ok=True)
    print(f"Restored {DATASET_MODE} data from Drive:", DRIVE_DATA)
    shutil.copytree(DRIVE_RESULTS, Path(LOCAL_RESULTS), dirs_exist_ok=True)
    if RESTORE_BASELINES and Path(DRIVE_BASELINES).is_dir():
        shutil.copytree(
            DRIVE_BASELINES,
            Path(MUSICINSTRUCT) / "results/baselines",
            dirs_exist_ok=True,
        )
        print("Restored baselines from Drive")

preds = f"{LOCAL_RESULTS}/predictions.jsonl"
print("dataset_mode:", DATASET_MODE)
print("manifest:", MANIFEST)
print("predictions:", preds, "exists:", Path(preds).is_file())

In [ ]:
# @title Rescore plan-executor predictions
import json
import subprocess
from pathlib import Path

if RESCORE_PLAN_EXECUTOR:
    pred_path = Path(LOCAL_RESULTS) / "predictions.jsonl"
    if not pred_path.is_file():
        raise FileNotFoundError(f"No predictions at {pred_path} — check Drive sync / RESULTS_SUBDIR")
    out = Path(LOCAL_RESULTS) / "score_report.json"
    manifest_rel = str(Path(MANIFEST).relative_to(Path(MUSICINSTRUCT)))
    cmd = [
        "musicinstruct",
        "score",
        manifest_rel,
        str(pred_path),
        "--split",
        SPLIT,
        "--output",
        str(out),
    ]
    print(" ".join(cmd))
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.stdout:
        print(proc.stdout)
    if proc.stderr:
        print(proc.stderr)
    if proc.returncode != 0:
        raise RuntimeError(
            f"musicinstruct score failed (exit {proc.returncode}). "
            "If stderr mentions 'bool is not JSON serializable', push latest main and re-run bootstrap."
        )
    report = json.loads(out.read_text())
    print("overall:", json.dumps(report.get("overall", {}), indent=2))
    print("composition_macro:", json.dumps(report.get("composition_macro", {}), indent=2))
else:
    print("Skipped rescore")

In [ ]:
# @title Run programmatic baselines
import json
from pathlib import Path

baseline_dir = Path(MUSICINSTRUCT) / "results/baselines"
if RUN_BASELINES:
    import subprocess

    manifest_rel = str(Path(MANIFEST).relative_to(Path(MUSICINSTRUCT)))
    cmd = [
        "musicinstruct",
        "run-baselines",
        manifest_rel,
        "--output-dir",
        str(baseline_dir),
        "--split",
        SPLIT,
    ]
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)
    if USE_DRIVE:
        import shutil

        shutil.copytree(baseline_dir, Path(DRIVE_BASELINES), dirs_exist_ok=True)
        print("Synced baselines →", DRIVE_BASELINES)

summary_path = baseline_dir / "baseline_report.json"
if summary_path.is_file():
    data = json.loads(summary_path.read_text())
    for name, block in data.get("baselines", {}).items():
        print(name, block.get("overall"))
elif not RUN_BASELINES:
    print("No baseline_report.json — set RUN_BASELINES=True or RESTORE_BASELINES=True")